## 1. API Connectivity

Test whether the individual recipe endpoint can be accessed directly using Python without browser automation or authentication.

In [ ]:
import requests

url = "https://jaukusskoniai.lt/api/recipes/virtu-bulviu-cepelinai-su-mesa"

response = requests.get(url, timeout=20)

print(response.status_code)
print(response.headers.get("Content-Type"))

## 2. JSON Structure Exploration

Inspect the top-level and nested JSON fields to identify relevant recipe metadata, Lithuanian content and ingredient information.

In [ ]:
data = response.json()

data.keys()

In [ ]:
data["content"].keys()

In [ ]:
data["content"]["lt"].keys()

## 3. Ingredient Exploration

Examine individual ingredient records, available attributes, original ingredient descriptions and ingredient-specific unit conversion factors.

In [ ]:
ingredients = data["content"]["ingredients"]

type(ingredients), len(ingredients)

In [ ]:
ingredients[0]

In [ ]:
import pandas as pd

df_ingredients = pd.DataFrame(ingredients)

df_ingredients.head()

In [ ]:
df_ingredients.columns

In [ ]:
ingredients[0]["original"]

## 4. Ingredient Quantity Normalisation

Convert ingredient quantities into grams using ingredient-specific conversion factors provided by the API. Preserve original quantities and units for grocery product matching.

Some conversion factors represent estimates rather than exact measured weights.

In [ ]:
ingredients[0]["units"]

In [ ]:
df_ingredients[["nameLt", "quantity", "unit", "grams", "defaultUnit"]]

In [ ]:
ingredient = ingredients[0]

recipe_unit = ingredient["unit"]

for conversion in ingredient["units"]:
    if conversion["unit"] == recipe_unit:
        grams_per_unit = conversion["gramsPerUnit"]
        break

print(grams_per_unit)

In [ ]:
total_grams = ingredient["quantity"] * grams_per_unit

print(total_grams)

In [ ]:
def get_grams_per_unit(ingredient):
    recipe_unit = ingredient["unit"]

    for conversion in ingredient["units"]:
        if conversion["unit"] == recipe_unit:
            return conversion["gramsPerUnit"]

    return None

In [ ]:
get_grams_per_unit(ingredients[0])

In [ ]:
df_ingredients["grams_per_unit"] = df_ingredients.apply(
    get_grams_per_unit, axis=1
)

In [ ]:
df_ingredients["total_grams"] = (
    df_ingredients["quantity"] * df_ingredients["grams_per_unit"]
)

In [ ]:
df_ingredients[
    ["nameLt", "quantity", "unit", "grams_per_unit", "total_grams"]
]

In [ ]:
df_ingredients["total_grams"].isna().sum()

### Findings

- All 15 ingredient entries were successfully converted into grams.
- No missing conversion results were identified in the sample recipe.
- Original measurement units should be preserved.
- Repeated ingredient IDs may represent ingredients used in different preparation stages and should not be removed as duplicates.
- Conversion coverage must be evaluated across additional recipes.

## 5. Recipe Data Extraction

Extract recipe-level metadata, including title, description, servings, preparation time, cooking time, cuisine, categories and instructions into a structured DataFrame.

In [ ]:
content = data["content"]

recipe = {
    "recipe_id": data["id"],
    "slug": data["slug"],
    "title": content["lt"]["title"],
    "description": content["lt"]["description"],
    "servings": content["servings"],
    "prep_minutes": content["prepMinutes"],
    "cook_minutes": content["cookMinutes"],
    "cuisine": content["cuisine"],
    "categories": [c["nameLt"] for c in data["categories"]],
    "instructions": [
        step["text"]
        for step in content["lt"]["steps"]
        if step["text"]
    ]
}

df_recipe = pd.DataFrame([recipe])

df_recipe

## 6. Linking Recipes and Ingredients

Assign the recipe identifier to each ingredient record to establish a one-to-many relationship between recipes and their ingredients. This will support subsequent relational database integration.

In [ ]:
df_ingredients["recipe_id"] = data["id"]

## 7. Catalogue and Pagination Exploration

Investigate the recipe catalogue endpoint and its cursor-based pagination mechanism to identify publicly accessible recipes for subsequent data collection.

In [ ]:
import time

def fetch_recipe_catalogue(max_pages=5, delay=2):
    all_recipes = []
    cursor = None

    for page in range(max_pages):
        params = {"cursor": cursor} if cursor else {}

        response = requests.get(
            catalogue_url,
            params=params,
            timeout=20
        )
        response.raise_for_status()

        result = response.json()
        recipes = result["recipes"]

        all_recipes.extend(recipes)

        print(f"Page {page + 1}: {len(recipes)} recipes")

        if not result["hasMore"]:
            break

        next_cursor = result.get("nextCursor")

        if not next_cursor or next_cursor == cursor:
            print("Pagination stopped: missing or repeated cursor")
            break

        cursor = next_cursor
        time.sleep(delay)

    return all_recipes

In [ ]:
catalogue_sample = fetch_recipe_catalogue(max_pages=5)

df_catalogue_sample = pd.DataFrame(catalogue_sample)

print("Total recipes:", len(df_catalogue_sample))
print("Unique recipes:", df_catalogue_sample["id"].nunique())
print("Duplicates:", df_catalogue_sample["id"].duplicated().sum())

In [ ]:
df_public = df_catalogue_sample[
    df_catalogue_sample["subscriberOnly"].eq(False)
].copy()

print("Public recipes:", len(df_public))
print("Excluded recipes:", len(df_catalogue_sample) - len(df_public))

In [ ]:
df_public[["id", "slug"]].head(10)

In [ ]:
sample_slugs = df_public["slug"].sample(
    n=min(10, len(df_public)),
    random_state=42
).tolist()

sample_slugs

In [ ]:
catalogue_url = "https://jaukusskoniai.lt/api/recipes"

response = requests.get(catalogue_url, timeout=20)

print(response.status_code)
print(response.headers.get("Content-Type"))

In [ ]:
catalogue = response.json()

catalogue.keys()

In [ ]:
df_catalogue = pd.DataFrame(catalogue["recipes"])

df_catalogue[["id", "slug", "subscriberOnly"]]

In [ ]:
df_catalogue["subscriberOnly"].value_counts()

In [ ]:
print("Page size:", catalogue["pageSize"])
print("Recipes returned:", len(catalogue["recipes"]))
print("Has more:", catalogue["hasMore"])
print("Next cursor:", catalogue["nextCursor"])

In [ ]:
cursor = catalogue["nextCursor"]

response = requests.get(
    catalogue_url,
    params={"cursor": cursor},
    timeout=20
)

print(response.status_code)

In [ ]:
catalogue_page_2 = response.json()

df_catalogue_page_2 = pd.DataFrame(
    catalogue_page_2["recipes"]
)

df_catalogue_page_2[["id", "slug", "subscriberOnly"]]

In [ ]:
page_1_ids = set(df_catalogue["id"])
page_2_ids = set(df_catalogue_page_2["id"])

overlap = page_1_ids.intersection(page_2_ids)

print("Overlapping recipes:", len(overlap))

## 8. Multiple Recipe Extraction and Validation

In [ ]:
def fetch_recipe(slug):
    url = f"{catalogue_url}/{slug}"

    response = requests.get(url, timeout=20)
    response.raise_for_status()

    data = response.json()

    if data.get("subscriberOnly") or data.get("locked"):
        return None

    return data

In [ ]:
def extract_recipe(data):
    content = data["content"]
    lt = content["lt"]
    nutrition = content.get("nutrition") or {}

    return {
        "recipe_id": data["id"],
        "slug": data["slug"],
        "title": lt["title"],
        "description": lt.get("description"),
        "servings": content.get("servings"),
        "prep_minutes": content.get("prepMinutes"),
        "cook_minutes": content.get("cookMinutes"),
        "cuisine": content.get("cuisine"),
        "categories": [
            c["nameLt"] for c in data.get("categories", [])
        ],
        "instructions": [
            step["text"]
            for step in lt.get("steps", [])
            if step.get("text")
        ],
        "kcal": nutrition.get("kcal"),
        "protein": nutrition.get("protein"),
        "fat": nutrition.get("fat"),
        "carbohydrates": nutrition.get("carbohydrates"),
        "sugars": nutrition.get("sugars"),
        "nutrition_complete": nutrition.get("complete"),
        "url": f"https://jaukusskoniai.lt/r/{data['slug']}"
    }

In [ ]:
def extract_ingredients(data):
    rows = []

    for position, ingredient in enumerate(
        data["content"].get("ingredients", [])
    ):
        original = ingredient.get("original") or {}

        quantity = ingredient.get("quantity")
        unit = ingredient.get("unit")

        conversion = next(
            (
                c for c in (ingredient.get("units") or [])
                if c["unit"] == unit
            ),
            {}
        )

        grams_per_unit = conversion.get("gramsPerUnit")

        total_grams = (
            quantity * grams_per_unit
            if quantity is not None and grams_per_unit is not None
            else None
        )

        rows.append({
            "recipe_id": data["id"],
            "position": position,
            "ingredient_id": ingredient.get("ingredientId"),
            "name_lt": ingredient.get("nameLt"),
            "original_name": original.get("name"),
            "notes": original.get("notes"),
            "group": original.get("group"),
            "quantity": quantity,
            "unit": unit,
            "default_unit": ingredient.get("defaultUnit"),
            "grams_per_unit": grams_per_unit,
            "total_grams": total_grams,
            "conversion_source": conversion.get("source"),
            "source_category": ingredient.get("sourceCategory"),
            "needs_review": ingredient.get("needsReview")
        })

    return rows

In [ ]:
recipe_rows = []
ingredient_rows = []

for slug in sample_slugs:
    data = fetch_recipe(slug)

    if data is not None:
        recipe_rows.append(extract_recipe(data))
        ingredient_rows.extend(extract_ingredients(data))

        print(f"Collected: {slug}")

    time.sleep(2)

df_recipes_sample = pd.DataFrame(recipe_rows)
df_ingredients_sample = pd.DataFrame(ingredient_rows)

print("\nRecipes:", len(df_recipes_sample))
print("Ingredient entries:", len(df_ingredients_sample))

In [ ]:
display(df_recipes_sample.head())
display(df_ingredients_sample.head())

In [ ]:
print("Recipes:", len(df_recipes_sample))

print("Unique recipes:",
      df_recipes_sample["recipe_id"].nunique())

print("Ingredient entries:",
      len(df_ingredients_sample))

print("Unique ingredients:",
      df_ingredients_sample["ingredient_id"].nunique())

print("Missing ingredient names:",
      df_ingredients_sample["name_lt"].isna().sum())

print("Missing quantities:",
      df_ingredients_sample["quantity"].isna().sum())

print("Missing weight conversions:",
      df_ingredients_sample["total_grams"].isna().sum())

print("\nMeasurement units:")
print(df_ingredients_sample["unit"].value_counts())

print("\nConversion sources:")
print(df_ingredients_sample["conversion_source"].value_counts())

## 9. Conclusions

The Jaukūs skoniai API provides structured Lithuanian
recipe data suitable for further development of Eat-easy.

Key findings:
- Individual recipes and catalogue pages can be retrieved directly using HTTP requests.
- Cursor-based pagination works without duplicate recipes in the tested sample.
- Recipe metadata and ingredients can be extracted into separate datasets.
- Ingredient identifiers support potential shopping list aggregation.
- All 118 ingredient entries across 10 sample recipes have valid names, quantities and available weight conversions.
- Approximately 76% of unit conversions are estimates.

Next step: implement a reusable recipe collection pipeline.